In [ ]:
%pip install -r requirements.txt

     |████████████████████████████████| 10.8 MB 8.0 MB/s eta 0:00:01
     |████████████████████████████████| 508 kB 27.8 MB/s eta 0:00:01
     |████████████████████████████████| 348 kB 41.6 MB/s eta 0:00:01
     |████████████████████████████████| 5.3 MB 9.4 MB/s eta 0:00:01
You should consider upgrading via the '/Users/prestonkou12/Desktop/School/TAMU/Fall2026/DAEN400/daen400_casestudies/daen400_venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [11]:
#### OS Packages to import excel file from local directory ####
import os
from pathlib import Path
from dotenv import load_dotenv
###############################################################

import pandas as pd 
import numpy as np
from openpyxl import load_workbook

In [20]:
#AI Generated Funcitons.
def is_empty_cell(val):
    """Check if a cell is empty or contains only whitespace."""
    if pd.isna(val):
        return True
    if isinstance(val, str) and val.strip() == '':
        return True
    return False

def get_contiguous_groups(bool_series):
    """Find contiguous True groups in a boolean series."""
    groups = []
    start = None
    
    bool_list = bool_series.tolist()
    
    for i, val in enumerate(bool_list):
        if val and start is None:
            start = i
        elif not val and start is not None:
            groups.append((start, i))
            start = None
    
    if start is not None:
        groups.append((start, len(bool_list)))
    
    return groups

def find_tables_in_sheet(df):
    """
    Finds all contiguous tables in a DataFrame (sheet).
    Tables are separated by completely empty rows/columns.
    """
    tables = []
    
    # Create mask using numpy for compatibility
    data = df.values
    mask = np.zeros(data.shape, dtype=bool)
    
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            mask[i, j] = not is_empty_cell(data[i, j])
    
    # Find row groups (rows that have at least one non-empty cell)
    row_has_data = mask.any(axis=1)
    row_groups = get_contiguous_groups(pd.Series(row_has_data))
    
    # For each row group, find column groups
    for row_start, row_end in row_groups:
        sub_mask = mask[row_start:row_end, :]
        col_has_data = sub_mask.any(axis=0)
        col_groups = get_contiguous_groups(pd.Series(col_has_data))
        
        # Extract each table
        for col_start, col_end in col_groups:
            table = df.iloc[row_start:row_end, col_start:col_end].copy()
            
            # Use first row as header
            new_columns = table.iloc[0].tolist()
            table = table.iloc[1:]
            table.columns = new_columns
            table = table.reset_index(drop=True)
            
            tables.append(table)
    
    return tables

def import_excel_tables(file_path):
    """
    Import all tables from all sheets in an Excel file.
    
    Returns:
        dict: {sheet_name: {table_1: df, table_2: df, ...}, ...}
    """
    all_tables = {}
    
    xlsx = pd.ExcelFile(file_path)
    
    for sheet_name in xlsx.sheet_names:
        df = pd.read_excel(
            file_path, 
            sheet_name=sheet_name, 
            header=None
        )
        
        tables = find_tables_in_sheet(df)
        
        sheet_tables = {}
        for i, table in enumerate(tables, 1):
            sheet_tables[f'table_{i}'] = table
        
        all_tables[sheet_name] = sheet_tables
    
    return all_tables


In [21]:
# ============ USAGE ============
env_path = Path('.') / '.env'
load_dotenv(dotenv_path=env_path)

DATASET_PATH = f"data/{os.getenv("DATASET_NAME")}"
all_tables = import_excel_tables(DATASET_PATH)

# View structure
#for sheet_name, tables in all_tables.items():
#    print(f"\nSheet: {sheet_name}")
#    for table_name, df in tables.items():
#        print(f"  {table_name}: {df.shape[0]} rows x {df.shape[1]} cols")

# Access a specific table
# df = all_tables['Sheet1']['table_1']
# print(df.head())